# TREC 2026 File Manifest Check

## tl;dr

This notebook reads `data/dev/file_manifest.tsv`, validates the expected columns and unique filenames, then shows which official TREC 2026 Million LLMs files are released versus still pending.

Expected result for the captured manifest: 5 rows total, 4 released files, and 1 test file marked `not_released`.

## Context & Methods

The input is the tracked manifest created from the official TREC 2026 Million LLMs Track page. The notebook does not download raw data. It only reads the local TSV manifest and reports the file inventory, source links, expected paths, and basic validation checks.

### Key Assumptions

- The notebook is run from this repository or a subdirectory inside it.
- Raw official data files may not exist locally yet; missing raw files are expected in a fresh checkout.
- `pandas` is available in the active Python environment.

## Data

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)


def find_repo_root(start: Path) -> Path:
    """Find the repo root by looking for the tracked TREC manifest."""
    for candidate in [start, *start.parents]:
        if (candidate / "data/dev/file_manifest.tsv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/dev/file_manifest.tsv from the current working directory.")


repo_root = find_repo_root(Path.cwd())
manifest_path = repo_root / "data/dev/file_manifest.tsv"
manifest_path

PosixPath('/home/npatta01/.codex/worktrees/0623/trec_mlm_2026/data/dev/file_manifest.tsv')

In [2]:
manifest = pd.read_csv(manifest_path, sep="	", dtype={"size_bytes": "Int64"})
manifest

,filename,split,status,size_bytes,google_drive_file_id,source_url,expected_local_path,purpose
0,llm_discovery_data.json,discovery,released,137300130,1H22vfnxsAQ7x4rox8QwSBUBhypOvz10U,https://drive.google.com/file/d/1H22vfnxsAQ7x4rox8QwSBUBhypOvz10U/view?usp=sharing,data/dev/llm_discovery_data.json,LLM expertise discovery queries and precomputed LLM responses.
1,llm_discovery_metadata.json,discovery,released,1460672563,1USvhIQZ8eMR5YNGt0VfIgNcisbneGBpO,https://drive.google.com/file/d/1USvhIQZ8eMR5YNGt0VfIgNcisbneGBpO/view?usp=sharing,data/dev/llm_discovery_metadata.json,Discovery queries and precomputed LLM responses with response metadata/logprobs.
2,llm_dev_data.tsv,development,released,40619,1t4lvw4uP4IS5JW474HNm3RdbSXG58Uf_,https://drive.google.com/file/d/1t4lvw4uP4IS5JW474HNm3RdbSXG58Uf_/view?usp=sharing,data/dev/llm_dev_data.tsv,Development query IDs and query text for ranking LLMs by expertise.
3,llm_dev_qrels.txt,development,released,2177782,1oq5i05khdxwKmI2HfDzDEuQIDkaM1KJ7,https://drive.google.com/file/d/1oq5i05khdxwKmI2HfDzDEuQIDkaM1KJ7/view?usp=sharing,data/dev/llm_dev_qrels.txt,Development qrels with expertise labels for query-LLM pairs.
4,llm_test_data.tsv,test,not_released,<NA>,TBD,TBD,data/dev/llm_test_data.tsv,Held-out test queries for official submission; expected release date 2026-09-05.


## Results

In [3]:
required_columns = {
    "filename",
    "split",
    "status",
    "size_bytes",
    "google_drive_file_id",
    "source_url",
    "expected_local_path",
    "purpose",
}

missing_columns = sorted(required_columns.difference(manifest.columns))
duplicate_filenames = manifest[manifest["filename"].duplicated(keep=False)]["filename"].tolist()

assert not missing_columns, f"Missing expected columns: {missing_columns}"
assert not duplicate_filenames, f"Duplicate filenames found: {duplicate_filenames}"

summary = pd.DataFrame(
    {
        "metric": [
            "rows",
            "released_files",
            "not_released_files",
            "unique_splits",
        ],
        "value": [
            len(manifest),
            int((manifest["status"] == "released").sum()),
            int((manifest["status"] == "not_released").sum()),
            manifest["split"].nunique(),
        ],
    }
)
summary

,metric,value
0,rows,5
1,released_files,4
2,not_released_files,1
3,unique_splits,3


In [4]:
status_by_split = (
    manifest.groupby(["split", "status"], as_index=False)
    .size()
    .rename(columns={"size": "file_count"})
    .sort_values(["split", "status"])
)
status_by_split

,split,status,file_count
0,development,released,2
1,discovery,released,2
2,test,not_released,1


In [5]:
released_files = manifest.loc[
    manifest["status"].eq("released"),
    ["filename", "split", "size_bytes", "expected_local_path", "source_url", "purpose"],
].reset_index(drop=True)
released_files

,filename,split,size_bytes,expected_local_path,source_url,purpose
0,llm_discovery_data.json,discovery,137300130,data/dev/llm_discovery_data.json,https://drive.google.com/file/d/1H22vfnxsAQ7x4rox8QwSBUBhypOvz10U/view?usp=sharing,LLM expertise discovery queries and precomputed LLM responses.
1,llm_discovery_metadata.json,discovery,1460672563,data/dev/llm_discovery_metadata.json,https://drive.google.com/file/d/1USvhIQZ8eMR5YNGt0VfIgNcisbneGBpO/view?usp=sharing,Discovery queries and precomputed LLM responses with response metadata/logprobs.
2,llm_dev_data.tsv,development,40619,data/dev/llm_dev_data.tsv,https://drive.google.com/file/d/1t4lvw4uP4IS5JW474HNm3RdbSXG58Uf_/view?usp=sharing,Development query IDs and query text for ranking LLMs by expertise.
3,llm_dev_qrels.txt,development,2177782,data/dev/llm_dev_qrels.txt,https://drive.google.com/file/d/1oq5i05khdxwKmI2HfDzDEuQIDkaM1KJ7/view?usp=sharing,Development qrels with expertise labels for query-LLM pairs.


In [6]:
def local_file_size(relative_path: str):
    path = repo_root / relative_path
    return path.stat().st_size if path.exists() else pd.NA


local_file_check = manifest[["filename", "status", "size_bytes", "expected_local_path"]].copy()
local_file_check["local_exists"] = local_file_check["expected_local_path"].map(
    lambda path: (repo_root / path).exists()
)
local_file_check["actual_size_bytes"] = (
    local_file_check["expected_local_path"].map(local_file_size).astype("Int64")
)
local_file_check["size_matches_manifest"] = (
    local_file_check["actual_size_bytes"] == local_file_check["size_bytes"]
)

local_file_check

,filename,status,size_bytes,expected_local_path,local_exists,actual_size_bytes,size_matches_manifest
0,llm_discovery_data.json,released,137300130,data/dev/llm_discovery_data.json,False,<NA>,<NA>
1,llm_discovery_metadata.json,released,1460672563,data/dev/llm_discovery_metadata.json,False,<NA>,<NA>
2,llm_dev_data.tsv,released,40619,data/dev/llm_dev_data.tsv,True,40619,True
3,llm_dev_qrels.txt,released,2177782,data/dev/llm_dev_qrels.txt,True,2177782,True
4,llm_test_data.tsv,not_released,<NA>,data/dev/llm_test_data.tsv,False,<NA>,<NA>


In [7]:
print(f"Manifest path: {manifest_path.relative_to(repo_root)}")
print(f"Rows loaded: {len(manifest)}")
print("Released files:")
for filename in released_files["filename"]:
    print(f"- {filename}")

pending = manifest.loc[manifest["status"].ne("released"), "filename"].tolist()
print("Pending files:", ", ".join(pending) if pending else "none")

Manifest path: data/dev/file_manifest.tsv
Rows loaded: 5
Released files:
- llm_discovery_data.json
- llm_discovery_metadata.json
- llm_dev_data.tsv
- llm_dev_qrels.txt
Pending files: llm_test_data.tsv


## Takeaways

- The manifest loads successfully and has the expected schema.
- Four official files are currently marked as released: two discovery files and two development files.
- The test file is represented as a placeholder because it was not released on the official page at capture time.
- The local raw-data check is expected to show missing files until the official files are downloaded into `data/dev/`.